##### *Read in current enhlib and barlib oPools for all STARR-FISH libraries*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import *

In [2]:
# Read in files containing oligo pools used to assemble CRE-barcode cassettes
enhlibV4 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\enhlibV4.xlsx')
barlibV4 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\barlibV4.xlsx')

enhlibV4 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\enhlibV4.xlsx')
barlibV4 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\barlibV4.xlsx')

enhlibV6 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\enhlibV6.xlsx')
barlibV6 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\barlibV6.xlsx')

enhlibV8 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\enhlibV8.xlsx')
barlibV8 = pd.read_excel(r'C:\Users\zgibbs\Documents\Oligo pools\barlibV8.xlsx')

##### *Swap AB positive control enhancers in to sublib 1 & 2*

In [8]:
from Bio.Seq import Seq

# create lists of current enhlib sublibraries #1 and #10 from SFv8
enhlib_01 = []

for i in range(40):
    enh_seq = Seq(enhlibV8["Sequence (5'-3')"][i])
    enhlib_01.append(str(enh_seq))

enhlib_10 = []

for i in np.arange(360,400):
    enh_seq = Seq(enhlibV8["Sequence (5'-3')"][i])
    enhlib_10.append(str(enh_seq))

In [12]:
from Bio import SeqIO

# load in fasta file of Allen Brain positive enhancer sequences
pos_ctrls = r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\final_pos_ctrls.fa'

pos_ctrl_seqs = []
for record in SeqIO.parse(pos_ctrls, "fasta"):
    pos_ctrl_seqs.append(str(record.seq.lower()))

In [13]:
# create new list of enhlib sublibraries using Allen Brain positive enhancer sequences
SFv8_01 = []

for i in range(40):
    fl_seq = enhlib_01[i][:23] + pos_ctrl_seqs[i] + enhlib_01[i][213:] # may need to change ranges depending on primer/overlap size
    SFv8_01.append(fl_seq)

SFv8_10 = []

for i in range(40):
    fl_seq = enhlib_10[i][:23] + pos_ctrl_seqs[i] + enhlib_10[i][213:] # may need to change ranges depending on primer/overlap size
    SFv8_10.append(fl_seq)

In [23]:
# load in bed file containing enhancer IDs
bed_file = pd.read_table(r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\final_pos_ctrls.bed', header=None)

In [31]:
# create dataframes containing the enhancer IDs and final oPool sequences
SFv8_01_pos_ctrl = pd.DataFrame(zip(bed_file.iloc[:,3], SFv8_01), columns=['enhancer ID', 'oPool sequence'])
SFv8_10_pos_ctrl = pd.DataFrame(zip(bed_file.iloc[:,3], SFv8_10), columns=['enhancer ID', 'oPool sequence'])

In [39]:
# save oPool files
SFv8_01_pos_ctrl.to_csv(r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\SFv8_01_pos_ctrls.csv', index=None)
SFv8_10_pos_ctrl.to_csv(r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\SFv8_10_pos_ctrls.csv', index=None)

##### *Pad the beginning of enhlib sequences that are Tm matched amongst each other in their overlap regions (from Bogdan)*

In [44]:
seqs_Tm = pd.read_csv(r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\SFv8_01_pos_ctrls_TMmatched.csv')['oPool sequence']
seqs_Tm[0]

'CATGGACGAGCTGTACAAGTAAGaaagctggaggggacaaaggatgctcaccttagaccaaggaagcagagccaagaagcacacatgcttgctgtgtaccataaatgcaatttttggaatgatgacaacagaaaagatgcaattcttagcatagactcagtcctgccaagagctgtctgacaatgtaggaaggattatttcaggaggaaaacaGCCACCTTAACACGCGATGAGGGTACATGCGCCTTACTCCGGATACATATACGCTCGTCGAGGGCCAAGTCGACCTAGATgcaatttgcgcttgttcggctaacgattttctcgcgggag'

In [69]:
# define the size difference between each enhlib sequence and the longest variant
sq_lengths = [len(seqs_Tm[i]) for i in np.arange(40)]
diffs = np.max(sq_lengths) - sq_lengths

In [64]:
# create function to generate random sequences
from Bio.SeqUtils import seq1
from Bio.Seq import Seq
import random

def generate_random_sequence(length):
    return ''.join(random.choices("atcg", k=length))

In [65]:
# create list of random sequences to pad with based on the difference in length between each enhlib sequence and the longest variant
random_sqs = []

for i in np.arange(40):
    sq = generate_random_sequence(diffs[i])
    random_sqs.append(sq)

In [73]:
# create new list of enhlib sequences with padded sequence added to the beginning
SFv8_01_Tm = []

for i in np.arange(40):
    fl_seq = random_sqs[i] + seqs_Tm[i] # may need to change ranges depending on primer/overlap size
    SFv8_01_Tm.append(fl_seq)

In [75]:
[len(SFv8_01_Tm[i]) for i in np.arange(40)]

[342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342,
 342]

In [77]:
# create dataframe containing the enhancer IDs and final Tm matched oPool sequences
SFv8_01_pos_ctrl_Tm = pd.DataFrame(zip(bed_file.iloc[:,3], SFv8_01_Tm), columns=['enhancer ID', 'oPool sequence'])

In [79]:
# save Tm matched oPool file
SFv8_01_pos_ctrl_Tm.to_csv(r'C:\Users\zgibbs\Allen_Brain_hall_of_fame_controls\SFv8_01_pos_ctrls_Tm.csv', index=None)